In [39]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [186]:
import pandas as pd
import os
import time
import dotenv
from tqdm import tqdm
import json

from llm import summarize_facts, summarize_law, extract_legal_principles, perform_citation_analysis
from llm import classify_key_case_importance, classify_three_levels_importance

from utils import save_json, load_json

from steps import do_stage_1, do_stage_2, do_stage_3, get_probabilities

from citation_analyzer.citation_extractor import analyze_citations

dotenv.load_dotenv()

input_data_root = '/Users/ahmed/Desktop/msc-24/ECHR/echr-processed'
output_dir = '/Users/ahmed/Desktop/msc-24/TND/workflow_v2.0/results'



In [41]:
def read_data():
    df_1 = pd.read_csv("/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_1.csv")
    list_1 = df_1["file_path"].to_list()

    df_2 = pd.read_csv("/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_2.csv")
    list_2 = df_2["file_path"].to_list()

    df_3 = pd.read_csv("/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_3.csv")
    list_3 = df_3["file_path"].to_list()

    df_4 = pd.read_csv("/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_4.csv")
    list_4 = df_4["file_path"].to_list()

    return list_1[:5], list_2[:5], list_3[:5], list_4[:5]

def check_processed_files():
    already_processed_files = os.listdir(output_dir)

    # remove the prefix 
    already_processed_files = [
        f.replace("i_1_case_", "") for f in already_processed_files
    ]
    already_processed_files = [
        f.replace("i_2_case_", "") for f in already_processed_files
    ]
    already_processed_files = [
        f.replace("i_3_case_", "") for f in already_processed_files
    ]
    already_processed_files = [
        f.replace("i_4_case_", "") for f in already_processed_files
    ]
    return already_processed_files

In [42]:
list_1, list_2, list_3, list_4 = read_data()

In [9]:
already_processed_files = check_processed_files()
for file_path in tqdm(list_2):
    # skip already processed files
    if file_path in already_processed_files:
        continue

    # read the case file
    print(f"Processing file: {file_path}")
    case_path = os.path.join(input_data_root, file_path)
    case_data = load_json(case_path)
    facts = case_data['facts']
    law = case_data['law']

    # Run the workflow
    print("Load facts.")
    facts = case_data['facts']
    print("Load law.")
    law = case_data['law']

    # Aggregate results and classify case importance
    inputs = {
        "facts": facts,
        "law": law,
    }

    # classify case importance 
    print("Classifying case importance.")
    #key_case_importance = classify_key_case_importance(inputs)
    three_levels_importance = classify_three_levels_importance(inputs)
    

    # save the results
    results = {
        "steps": {
            "facts": facts,
            "law": law,
        },
        "three_levels_importance": three_levels_importance,
        "ground_truth": case_data['importance']
    }
    output_file_name = f'i_{case_data["importance"]}_case_{file_path}'
    save_json(os.path.join(output_dir, output_file_name), results)
    time.sleep(3)


  0%|          | 0/5 [00:00<?, ?it/s]

Processing file: 001-211127.json
Load facts.
Load law.
Classifying case importance.


 20%|██        | 1/5 [00:09<00:39,  9.78s/it]

Processing file: 001-211018.json
Load facts.
Load law.
Classifying case importance.


 40%|████      | 2/5 [00:19<00:29,  9.77s/it]

...

 60%|██████    | 3/5 [00:31<00:21, 10.81s/it]

Processing file: 001-209033.json
Load facts.
Load law.
Classifying case importance.


 80%|████████  | 4/5 [00:39<00:09,  9.47s/it]

Processing file: 001-207385.json
Load facts.
Load law.
Classifying case importance.


100%|██████████| 5/5 [00:46<00:00,  9.31s/it]


In [15]:
results

{'steps': {'facts': '2. The applicant was born in 1973 and is serving his life sentence in the Izyaslav Correctional Colony, Ukraine. The applicant was represented by Ms O. O. Protsenko, Ms A. G. Kozmenko, Ms V. P. Lebid and Mr M. O. Tarakhkalo, lawyers practising in Kyiv, Ukraine.\n3. The Government were represented by their Acting Agent, Ms O.V. Davydchuk, from the Ministry of Justice.\n4. The facts of the case may be summarised as follows.\n5. On 8 December 1999 the applicant was arrested in Hungary on suspicion of having committed a double murder with Mr K.\n. On 21 March 2002 the B k s District Court (Hungary) found the applicant and his accomplice K. guilty of conspiracy to commit premeditated double murder for motives of personal gain, the offence being committed on 11 November 1999. The applicant was sentenced to life imprisonment with the possibility of release on parole after serving twenty years of imprisonment and being deported.\n. On 21 August 2003 the Szeged Court of App

In [215]:
# load the results
high_importance_files = []
result_dir = '/Users/ahmed/Desktop/msc-24/TND/three_levels_classification/simple_CoT/results_facts_law'
files = os.listdir(result_dir)
for file in files:
    data = load_json(os.path.join(result_dir, file))
    if 'HIGH' in data['result']['classification_reasoning']:
        high_importance_files.append(file)

In [216]:
len(high_importance_files)

261

In [217]:
for name in high_importance_files:
    print(name)

i_3_case_001-218302.json
i_2_case_001-122893.json
i_3_case_001-217624.json
i_2_case_001-141781.json
i_2_case_001-146044.json
i_3_case_001-219786.json
i_2_case_001-119055.json
i_3_case_001-219944.json
i_3_case_001-219102.json
i_3_case_001-217370.json
i_2_case_001-139908.json
i_3_case_001-217537.json
i_4_case_001-229421.json
i_2_case_001-109741.json
i_3_case_001-220890.json
i_2_case_001-163222.json
i_3_case_001-219641.json
i_2_case_001-138948.json
i_4_case_001-228676.json
i_3_case_001-218070.json
i_2_case_001-206212.json
i_2_case_001-102711.json
i_2_case_001-108465.json
i_3_case_001-217805.json
i_3_case_001-219773.json
i_2_case_001-112091.json
i_3_case_001-217716.json
i_2_case_001-109034.json
i_2_case_001-112013.json
i_2_case_001-139084.json
i_3_case_001-223024.json
i_3_case_001-219070.json
i_2_case_001-111690.json
i_3_case_001-222789.json
i_4_case_001-228840.json
i_3_case_001-225220.json
i_3_case_001-220537.json
i_2_case_001-113300.json
i_3_case_001-225762.json
i_2_case_001-105217.json


In [222]:
file = 'i_4_case_001-229425.json'
data = load_json(os.path.join('/Users/ahmed/Desktop/msc-24/TND/workflow_v2.0/results', file))
facts = data['steps']['facts']
law = data['steps']['law']
three_levels_importance = data['three_levels_importance']

normalized_file_name = file.replace("i_1_case_", "")
normalized_file_name = normalized_file_name.replace("i_2_case_", "")
normalized_file_name = normalized_file_name.replace("i_3_case_", "")
normalized_file_name = normalized_file_name.replace("i_4_case_", "")
#citation_analysis = analyze_citations(normalized_file_name)
case_data = load_json(os.path.join(input_data_root, normalized_file_name))


case_name = case_data['docname']
facts = case_data['facts']
law = case_data['law']
articles = case_data['__articles']
#stage_1 = do_stage_1(articles)
#stage_2 = do_stage_2(case_name, facts, law, stage_1)
#stage_3 = do_stage_3(case_name, facts, law, stage_1, stage_2)
probabilities = get_probabilities(case_name, facts, law, citation_analysis, stage_1, stage_2, stage_3)

data.update({'citation_analysis': citation_analysis})
data.update({'stage_1': stage_1, 'stage_2': stage_2, 'stage_3': stage_3, 'probabilities': probabilities})

save_json(os.path.join('/Users/ahmed/Desktop/msc-24/TND/workflow_v2.0/results_s2', file), data)


In [233]:
stage_1

'Certainly! Below is an analysis of the relevant articles from the European Convention on Human Rights (ECHR) as they pertain to your request. Each article is broken down into established tests, key principles, standard interpretations, and typical application contexts, following your example format.\n\n```json\n{\n    "article_2": {\n        "article_number": "2",\n        "established_tests": [\n            {\n                "test_name": "Right to Life",\n                "key_elements": ["Protection against unlawful deprivation of life", "Obligation to investigate deaths", "Use of lethal force"],\n                "source_cases": ["McCann and Others v. the United Kingdom", "Öneryıldız v. Turkey"]\n            }\n        ],\n        "core_principles": [\n            {\n                "principle": "The state\'s duty to protect life",\n                "established_by": "Osman v. the United Kingdom",\n                "year_established": 1998\n            }\n        ],\n        "standard

In [223]:
probabilities = probabilities.replace('```json', '').replace('```', '')
probabilities = probabilities.strip()
probabilities_json = json.loads(probabilities)
probabilities_json

{'justifications': {'Key Case': 'The case of Doronin and Others v. Russia does not establish new legal principles or significantly redefine existing ones. It largely follows established jurisprudence regarding the right to freedom of assembly under Article 11, reinforcing prior rulings without introducing novel interpretations. While it does address systemic issues related to the application of these rights in a context of increasing state repression in Russia, the case does not adequately resolve critical gaps or ambiguities in ECHR jurisprudence. The relevance of the findings is more about clarifying and reaffirming existing standards than creating new pathways for future cases.',
  'Not Key Case': 'The judgments primarily build upon existing cases and legal principles concerning the right to freedom of assembly, notably drawing on previous cases such as Frumkin and Navalnyy. The findings concerning procedural fairness and proportionality do not introduce groundbreaking interpretatio

In [225]:
file = 'i_2_case_001-207385.json'
data = load_json(os.path.join('/Users/ahmed/Desktop/msc-24/TND/workflow_v2.0/results', file))
facts = data['steps']['facts']
law = data['steps']['law']
three_levels_importance = data['three_levels_importance']

normalized_file_name = file.replace("i_1_case_", "")
normalized_file_name = normalized_file_name.replace("i_2_case_", "")
normalized_file_name = normalized_file_name.replace("i_3_case_", "")
normalized_file_name = normalized_file_name.replace("i_4_case_", "")
citation_analysis = analyze_citations(normalized_file_name)
case_data = load_json(os.path.join(input_data_root, normalized_file_name))


case_name = case_data['docname']
facts = case_data['facts']
law = case_data['law']
articles = case_data['__articles']
stage_1 = do_stage_1(articles)
stage_2 = do_stage_2(case_name, facts, law, stage_1)
stage_3 = do_stage_3(case_name, facts, law, stage_1, stage_2)
probabilities = get_probabilities(case_name, facts, law, citation_analysis, stage_1, stage_2, stage_3)


001-59590
001-114492
001-73312
001-92169
001-80455
001-71673
001-75221
001-175646
001-86885
001-155198
001-159778
001-83273
001-58901
001-115657
001-94162
001-58735


In [228]:
print(probabilities)

{
    "justifications": {
        "Key Case": "The case establishes critical connections between criminal convictions and civil claims for damages in the context of forced disappearances, indicating a potential shift in how courts assess causal links in human rights violations. This establishes new legal principles and clarifies ambiguities in ECHR jurisprudence regarding state accountability and procedural fairness in civil proceedings by incorporating findings from criminal judgments into civil liability. Moreover, it has broad implications for future cases pertaining to state actions during conflicts, signifying a systemic change in legal standards, especially concerning access to justice.",
        "Not Key Case": "Despite its significant implications, the case largely evolves existing jurisprudential interpretations rather than introducing entirely new legal frameworks. The principles touched upon have precedents, and even though it clarifies important causal links in a nuanced co

In [231]:
import re
def parse_json(response):
    json_match = re.search(r"{.*?}", response, re.DOTALL)
    if json_match:
        json_str = json_match.group()
        try:
            probabilities_json = json.loads(json_str)
            print(probabilities_json)
        except json.JSONDecodeError as e:
            print("Error decoding JSON:", e)
    else:
        print("No JSON object found in the response.")

In [232]:
parse_json(probabilities)

Error decoding JSON: Expecting ',' delimiter: line 5 column 6 (char 1226)


In [226]:
probabilities = probabilities.replace('```json', '').replace('```', '')
probabilities = probabilities.strip()
probabilities_json = json.loads(probabilities)
probabilities_json

JSONDecodeError: Extra data: line 13 column 1 (char 1313)

In [193]:
file = 'i_1_case_001-220960.json'
data = load_json(os.path.join(result_dir, file))
facts = data['steps']['facts']
law = data['steps']['law']
three_levels_importance = data['three_levels_importance']

normalized_file_name = file.replace("i_1_case_", "")
citation_analysis = analyze_citations(normalized_file_name)
case_data = load_json(os.path.join(input_data_root, normalized_file_name))


case_name = case_data['docname']
facts = case_data['facts']
law = case_data['law']
articles = case_data['__articles']
stage_1 = do_stage_1(articles)
stage_2 = do_stage_2(case_name, facts, law, stage_1)
stage_3 = do_stage_3(case_name, facts, law, stage_1, stage_2)
probabilities = get_probabilities(case_name, facts, law, citation_analysis, stage_1, stage_2, stage_3)


001-122664
001-150778
001-85019
001-127697
001-126025
001-57927
001-89161
001-97425
001-103663
001-76766
001-173776
001-163671


In [194]:
probabilities = probabilities.replace('```json', '').replace('```', '')
probabilities = probabilities.strip()
probabilities_json = json.loads(probabilities)
probabilities_json

{'probabilities': {'Key Case': 80, 'Not Key Case': 20},
 'justifications': {'Key Case': "The case of Kupinskiy v. Ukraine introduces significant legal implications regarding the conversion of a reducible life sentence into a de facto irreducible life sentence without a clear legal framework for parole. This situation sheds light on systemic issues within Ukrainian law, aligning with Article 3 as it addresses inhuman treatment tied to the lack of a parole mechanism. Furthermore, the Court's assessment highlights potential violations under Article 7 concerning the definition and scope of penalties when transferring sentences across jurisdictions. These elements, combined with the case's reliance on established principles from past significant rulings (such as Vinter and Hutchinson), lead to the conclusion that it does expand the scope of ECHR jurisprudence, contributing to a broader understanding of how life sentences, particularly in cross-jurisdictional contexts, must be treated under 

In [234]:
s = "{\n    \"article_baseline\": {\n        \"article_5\": {\n            \"article_number\": \"5\",\n            \"established_tests\": [\n                {\n                    \"test_name\": \"Lawfulness of Detention\",\n                    \"key_elements\": [\n                        \"Grounds for detention must be clearly stated in law\",\n                        \"Detention must be necessary and proportionate\",\n                        \"Safeguards against arbitrary detention must be in place\"\n                    ]\n                }\n            ],\n            \"core_principles\": [\n                {\n                    \"principle\": \"Right to Liberty and Security\",\n                    \"year_established\": 1950\n                }\n            ],\n            \"standard_interpretations\": [\n                {\n                    \"interpretation\": \"Detention must follow due process, and individuals must be informed of the reasons for their detention.\"\n                }\n            ]\n        },\n\n        \"article_6\": {\n            \"article_number\": \"6\",\n            \"established_tests\": [\n                {\n                    \"test_name\": \"Right to a Fair Trial\",\n                    \"key_elements\": [\n                        \"Public hearing\",\n                        \"Impartial tribunal\",\n                        \"Equality of arms\",\n                        \"Right to legal assistance\"\n                    ]\n                }\n            ],\n            \"core_principles\": [\n                {\n                    \"principle\": \"Fair Hearing\",\n                    \"year_established\": 1950\n                }\n            ],\n            \"standard_interpretations\": [\n                {\n                    \"interpretation\": \"The right to mount a defense, including the right to legal counsel, is fundamental to fair judicial proceedings.\"\n                }\n            ]\n        },\n\n        \"article_10\": {\n            \"article_number\": \"10\",\n            \"established_tests\": [\n                {\n                    \"test_name\": \"Freedom of Expression\",\n                    \"key_elements\": [\n                        \"Intrusions must be provided by law\",\n                        \"Legitimate aim for restriction must be established\",\n                        \"Restrictions must be necessary in a democratic society\"\n                    ]\n                }\n            ],\n            \"core_principles\": [\n                {\n                    \"principle\": \"Right to Freedom of Expression\",\n                    \"year_established\": 1950\n                }\n            ],\n            \"standard_interpretations\": [\n                {\n                    \"interpretation\": \"Freedom of expression includes not only spoken and written words but also information and ideas regardless of form.\"\n                }\n            ]\n        },\n\n        \"article_11\": {\n            \"article_number\": \"11\",\n            \"established_tests\": [\n                {\n                    \"test_name\": \"Freedom of Assembly and Association\",\n                    \"key_elements\": [\n                        \"Right to peaceful assembly\",\n                        \"Regulations or restrictions must be necessary in a democratic society\",\n                        \"Freedom to form and join associations\"\n                    ]\n                }\n            ],\n            \"core_principles\": [\n                {\n                    \"principle\": \"Right to Join Trade Unions and Associations\",\n                    \"year_established\": 1950\n                }\n            ],\n            \"standard_interpretations\": [\n                {\n                    \"interpretation\": \"The right to assemble peacefully is fundamental, and any interference must be justified as necessary.\"\n                }\n            ]\n        },\n\n        \"protocol_7_article_2\": {\n            \"article_number\": \"P7-2\",\n            \"established_tests\": [\n                {\n                    \"test_name\": \"Right to Compensation\",\n                    \"key_elements\": [\n                        \"Provision of an effective remedy\",\n                        \"Compensation for wrongful conviction\",\n                        \"Timeliness and adequacy of redress\"\n                    ]\n                }\n            ],\n            \"core_principles\": [\n                {\n                    \"principle\": \"Right to Compensation for Unlawful Detention\",\n                    \"year_established\": 1984\n                }\n            ],\n            \"standard_interpretations\": [\n                {\n                    \"interpretation\": \"Individuals wrongfully convicted have the right to adequate compensation, reflecting the severity of the error.\"\n                }\n            ]\n        }\n    }\n}"

ss = json.loads(s)

In [235]:
ss

{'article_baseline': {'article_5': {'article_number': '5',
   'established_tests': [{'test_name': 'Lawfulness of Detention',
     'key_elements': ['Grounds for detention must be clearly stated in law',
      'Detention must be necessary and proportionate',
      'Safeguards against arbitrary detention must be in place']}],
   'core_principles': [{'principle': 'Right to Liberty and Security',
     'year_established': 1950}],
   'standard_interpretations': [{'interpretation': 'Detention must follow due process, and individuals must be informed of the reasons for their detention.'}]},
  'article_6': {'article_number': '6',
   'established_tests': [{'test_name': 'Right to a Fair Trial',
     'key_elements': ['Public hearing',
      'Impartial tribunal',
      'Equality of arms',
      'Right to legal assistance']}],
   'core_principles': [{'principle': 'Fair Hearing',
     'year_established': 1950}],
   'standard_interpretations': [{'interpretation': 'The right to mount a defense, includin

In [236]:
import tiktoken